# 📖 Notebook 1: Geospatial Matching

When a rider requests a ride, we need to find the **closest available drivers** fast. This is a proximity search — given a point on Earth, find all points within X kilometers.

Regular database indexes (B-trees) are terrible at this because latitude and longitude are **two dimensions**. Searching both at once requires a full table scan or ugly range queries.

This notebook shows two solutions:
1. **PostGIS** — a PostgreSQL extension with spatial indexes (R-trees) built for this
2. **Redis Geo** — an in-memory geospatial index for ultra-fast lookups

## Learning Objectives

By the end of this notebook, you'll understand:
- Why regular indexes fail for location queries
- How PostGIS uses spatial indexes to find nearby points efficiently
- How Redis Geo uses geohashing for in-memory proximity searches
- When to use each approach (and why Uber uses both)

## 🛠️ Setup

Start the infrastructure first:

```bash
cd system-designs/uber
docker-compose up -d
```

### Visualization Tools

- **Adminer** (PostgreSQL GUI): http://localhost:8080  
  Login: System `PostgreSQL`, Server `postgres`, User `demo`, Password `demo`, Database `uber_demo`
- **RedisInsight** (Redis GUI): http://localhost:5540  
  Click "Add Redis Database" → Host `redis`, Port `6379`

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import redis
import time
import json

# Database connection settings
DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "uber_demo",
    "user": "demo",
    "password": "demo"
}

# Redis connection settings
REDIS_CONFIG = {
    "host": "localhost",
    "port": 6379,
    "decode_responses": True
}

def get_db():
    return psycopg2.connect(**DB_CONFIG)

def get_redis():
    return redis.Redis(**REDIS_CONFIG)

# Test both connections
try:
    conn = get_db()
    conn.close()
    print("✅ Connected to PostgreSQL + PostGIS")
except Exception as e:
    print(f"❌ PostgreSQL failed: {e}")
    print("   Run: docker-compose up -d")

try:
    r = get_redis()
    r.ping()
    print("✅ Connected to Redis")
except Exception as e:
    print(f"❌ Redis failed: {e}")
    print("   Run: docker-compose up -d")

## 🤔 The Problem: Finding Nearby Drivers

Imagine you're standing in downtown San Francisco and you tap "Request Ride."  
The system needs to find all available drivers within, say, 3 km of you.

Each driver has a latitude and longitude. Your location is also a lat/lng pair.  
The question is: **which drivers are close to me?**

### Why Regular SQL Fails

A naive approach might be:
```sql
SELECT * FROM drivers 
WHERE ABS(lat - my_lat) < 0.03 
  AND ABS(lng - my_lng) < 0.03;
```

Problems:
1. **Inaccurate** — longitude degrees vary in distance depending on latitude
2. **Slow** — B-tree indexes can't efficiently search two columns at once
3. **No Earth curvature** — the Earth is round, distances aren't just Pythagoras

We need **spatial indexes** — data structures designed for multi-dimensional data.

## 🐌 Approach 1: Naive Python Scan (the bad baseline)

Before we reach for fancy spatial tools, let's try the obvious thing a beginner would write:

1. Fetch **all** drivers from the database.
2. Loop over them in Python and compute the distance to the rider.
3. Sort by distance, keep the closest 5.

This is the **O(N) per query** approach. It works fine for 10 drivers — and falls over at 5 million.
We compute distance with the **Haversine formula**, which accounts for the Earth's curvature.


In [ ]:
# Approach 1 — naive: scan every driver in Python
import math

def haversine_km(lat1, lng1, lat2, lng2):
    """Great-circle distance between two lat/lng points, in kilometers."""
    R = 6371.0  # Earth radius in km
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp = math.radians(lat2 - lat1)
    dl = math.radians(lng2 - lng1)
    a = math.sin(dp/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    return 2 * R * math.asin(math.sqrt(a))

rider_lat, rider_lng = 37.7749, -122.4194

conn = get_db(); cur = conn.cursor()
# Fetch EVERY driver (no spatial filter at the DB level — the bad part)
cur.execute("""
    SELECT d.id, d.name, d.status,
           ST_Y(dl.location::geometry) AS lat,
           ST_X(dl.location::geometry) AS lng
    FROM drivers d JOIN driver_locations dl ON d.id = dl.driver_id;
""")
rows = cur.fetchall()
conn.close()

# Compute distance in Python for each row, filter available, sort
scored = [
    (did, name, haversine_km(rider_lat, rider_lng, lat, lng))
    for did, name, status, lat, lng in rows
    if status == "available"
]
scored.sort(key=lambda x: x[2])

print(f"Scanned {len(rows)} drivers in Python, kept 5 nearest:")
for did, name, d in scored[:5]:
    print(f"  driver:{did:<3} {name:<16} {d:.2f} km")

print()
print("⚠️  Why this is the 'bad' baseline:")
print("   • Transfers every driver row over the wire on every request.")
print("   • Does the math in Python, not the database.")
print("   • Cost grows linearly with total drivers — at 5M drivers,")
print("     a single rider tap would read millions of rows. Not survivable.")
print("   Next: let the database prune by location using a spatial index.")


## 🗺️ Approach 2: PostGIS (Spatial Index)

PostGIS adds geographic data types to PostgreSQL. Instead of storing lat/lng as two numbers, we store a **GEOGRAPHY(POINT)** — a proper geospatial object.

PostGIS then uses a **GiST index** (Generalized Search Tree) which is like a B-tree but for shapes and regions. It divides space into bounding boxes, so "find all points within 3 km" only checks the relevant boxes.

Let's see it in action.

In [ ]:
# Let's see what drivers are in our database and where they are

conn = get_db()
cur = conn.cursor()

cur.execute("""
    SELECT 
        d.id,
        d.name,
        d.status,
        d.vehicle_make || ' ' || d.vehicle_model AS vehicle,
        ST_Y(dl.location::geometry) AS latitude,
        ST_X(dl.location::geometry) AS longitude
    FROM drivers d
    JOIN driver_locations dl ON d.id = dl.driver_id
    ORDER BY d.id;
""")

print(f"{'ID':<4} {'Name':<16} {'Status':<12} {'Vehicle':<20} {'Lat':>10} {'Lng':>12}")
print("-" * 80)
for row in cur.fetchall():
    print(f"{row[0]:<4} {row[1]:<16} {row[2]:<12} {row[3]:<20} {row[4]:>10.4f} {row[5]:>12.4f}")

conn.close()

In [ ]:
# 🎯 Find the 5 nearest AVAILABLE drivers to a rider in downtown SF
#
# Rider location: downtown San Francisco (-122.4194, 37.7749)
# We use ST_DWithin to filter by distance, then ST_Distance to sort.

rider_lng = -122.4194
rider_lat = 37.7749
search_radius_meters = 5000  # 5 km

conn = get_db()
cur = conn.cursor()

cur.execute("""
    SELECT 
        d.id,
        d.name,
        d.vehicle_make || ' ' || d.vehicle_model AS vehicle,
        ROUND(ST_Distance(
            dl.location,
            ST_SetSRID(ST_MakePoint(%s, %s), 4326)::geography
        )::numeric, 0) AS distance_meters
    FROM drivers d
    JOIN driver_locations dl ON d.id = dl.driver_id
    WHERE d.status = 'available'
      AND ST_DWithin(
            dl.location,
            ST_SetSRID(ST_MakePoint(%s, %s), 4326)::geography,
            %s  -- radius in meters
          )
    ORDER BY distance_meters
    LIMIT 5;
""", (rider_lng, rider_lat, rider_lng, rider_lat, search_radius_meters))

print(f"🎯 Nearest available drivers to ({rider_lat}, {rider_lng}):")
print(f"   Search radius: {search_radius_meters / 1000} km")
print()
print(f"{'ID':<4} {'Name':<16} {'Vehicle':<20} {'Distance':>10}")
print("-" * 55)
for row in cur.fetchall():
    dist_km = float(row[3]) / 1000
    print(f"{row[0]:<4} {row[1]:<16} {row[2]:<20} {dist_km:>8.2f} km")

conn.close()

In [ ]:
# Let's see how the spatial index helps with EXPLAIN ANALYZE

conn = get_db()
cur = conn.cursor()

cur.execute("""
    EXPLAIN ANALYZE
    SELECT d.id, d.name
    FROM drivers d
    JOIN driver_locations dl ON d.id = dl.driver_id
    WHERE d.status = 'available'
      AND ST_DWithin(
            dl.location,
            ST_SetSRID(ST_MakePoint(-122.4194, 37.7749), 4326)::geography,
            5000
          );
""")

print("📊 Query Plan (notice the GiST index scan):")
print()
for row in cur.fetchall():
    print(f"  {row[0]}")

print()
print("💡 The GiST index lets PostGIS skip most of the table.")
print("   Without it, every row would need a distance calculation.")

conn.close()

## ⚡ Approach 3: Redis Geo (In-Memory Speed)

PostGIS is great for durable storage, but it lives on disk. With 2 million location updates per second, we need something faster.

**Redis Geo** stores locations in memory using **geohashing** — it converts (lat, lng) into a single integer that preserves spatial locality. Nearby points have similar geohash values, so Redis can find neighbors efficiently using its sorted set data structure.

Key commands:
- `GEOADD key lng lat member` — add/update a location
- `GEOSEARCH key FROMLONLAT lng lat BYRADIUS 5 km` — find nearby members
- `GEODIST key member1 member2 km` — distance between two members

In [ ]:
# Load all driver locations from PostGIS into Redis Geo
# In production, drivers send updates directly to Redis

r = get_redis()
conn = get_db()
cur = conn.cursor()

cur.execute("""
    SELECT 
        d.id,
        d.name,
        d.status,
        ST_X(dl.location::geometry) AS longitude,
        ST_Y(dl.location::geometry) AS latitude
    FROM drivers d
    JOIN driver_locations dl ON d.id = dl.driver_id;
""")

# Clear any existing data
r.delete("drivers:locations")
r.delete("drivers:available")

for row in cur.fetchall():
    driver_id, name, status, lng, lat = row
    
    # GEOADD adds the driver's location to a geo set
    r.geoadd("drivers:locations", (lng, lat, f"driver:{driver_id}"))
    
    # Track available drivers in a separate set for fast filtering
    if status == "available":
        r.sadd("drivers:available", f"driver:{driver_id}")
    
    print(f"  Added driver:{driver_id} ({name}) at ({lat:.4f}, {lng:.4f}) — {status}")

conn.close()
print()
print(f"✅ Loaded {r.zcard('drivers:locations')} drivers into Redis Geo")
print(f"   {r.scard('drivers:available')} are marked available")

In [ ]:
# 🎯 Find nearest drivers using Redis GEOSEARCH
# This is what happens in real time when a rider requests a ride

rider_lng = -122.4194
rider_lat = 37.7749
search_radius_km = 5

# GEOSEARCH returns members within radius, sorted by distance
nearby = r.geosearch(
    name="drivers:locations",
    longitude=rider_lng,
    latitude=rider_lat,
    radius=search_radius_km,
    unit="km",
    withcoord=True,
    withdist=True,
    sort="ASC",  # nearest first
    count=10
)

# Get the set of available drivers
available = r.smembers("drivers:available")

print(f"🎯 Redis GEOSEARCH: drivers within {search_radius_km} km of downtown SF")
print()
print(f"{'Driver':<14} {'Distance':>10} {'Available':>10} {'Coordinates':>24}")
print("-" * 62)
for member, dist, coords in nearby:
    is_available = "✅ yes" if member in available else "❌ no"
    print(f"{member:<14} {dist:>8.2f} km {is_available:>10}   ({coords[1]:.4f}, {coords[0]:.4f})")

# Filter to only available drivers
matches = [(m, d) for m, d, c in nearby if m in available]
print()
print(f"🚗 Best match: {matches[0][0]} at {matches[0][1]:.2f} km away" if matches else "❌ No available drivers nearby!")

In [ ]:
# ⏱️ Speed comparison: PostGIS vs Redis Geo

iterations = 100

# Measure PostGIS
conn = get_db()
postgis_times = []
for _ in range(iterations):
    cur = conn.cursor()
    start = time.time()
    cur.execute("""
        SELECT d.id
        FROM drivers d
        JOIN driver_locations dl ON d.id = dl.driver_id
        WHERE d.status = 'available'
          AND ST_DWithin(
                dl.location,
                ST_SetSRID(ST_MakePoint(-122.4194, 37.7749), 4326)::geography,
                5000)
        ORDER BY ST_Distance(
                dl.location,
                ST_SetSRID(ST_MakePoint(-122.4194, 37.7749), 4326)::geography)
        LIMIT 5;
    """)
    cur.fetchall()
    postgis_times.append((time.time() - start) * 1000)
conn.close()

# Measure Redis Geo
redis_times = []
for _ in range(iterations):
    start = time.time()
    r.geosearch(
        name="drivers:locations",
        longitude=-122.4194, latitude=37.7749,
        radius=5, unit="km",
        withdist=True, sort="ASC", count=5
    )
    redis_times.append((time.time() - start) * 1000)

avg_postgis = sum(postgis_times) / len(postgis_times)
avg_redis = sum(redis_times) / len(redis_times)

print(f"⏱️ Proximity Search Latency ({iterations} runs each):")
print(f"{'':>4}{'Avg':>10}{'Min':>10}{'Max':>10}")
print(f"  PostGIS: {avg_postgis:>8.2f}ms {min(postgis_times):>8.2f}ms {max(postgis_times):>8.2f}ms")
print(f"  Redis:   {avg_redis:>8.2f}ms {min(redis_times):>8.2f}ms {max(redis_times):>8.2f}ms")
print()
print(f"🚀 Redis is {avg_postgis / avg_redis:.1f}× faster for proximity searches!")
print()
print("💡 This is why Uber uses Redis for real-time matching")
print("   and PostGIS/similar for analytics and historical data.")

## 🧠 How Geohashing Works (Behind Redis Geo)

Redis doesn't store raw lat/lng — it converts them into a **geohash**, a single integer that encodes both coordinates. Here's the key insight:

```
Geohashing divides the world into a grid of cells.
Each cell gets a unique code. The longer the code,
the smaller (more precise) the cell.

┌────────┬────────┐
│  00    │  01    │   2-bit geohash: 4 cells
├────────┼────────┤
│  10    │  11    │
└────────┴────────┘

┌────┬────┬────┬────┐
│0000│0001│0100│0101│
├────┼────┼────┼────┤  4-bit geohash: 16 cells
│0010│0011│0110│0111│
├────┼────┼────┼────┤
│1000│1001│1100│1101│
├────┼────┼────┼────┤
│1010│1011│1110│1111│
└────┴────┴────┴────┘
```

Points in the same cell or adjacent cells share a **common prefix**.  
Redis stores geohashes in a sorted set — nearby points are near each other in the sorted order.

To find neighbors, Redis looks at a small range of the sorted set instead of scanning everything.

In [ ]:
# You can retrieve the geohash for any member

for driver_id in range(1, 6):
    member = f"driver:{driver_id}"
    geohash = r.geohash("drivers:locations", member)
    pos = r.geopos("drivers:locations", member)
    
    if geohash[0] and pos[0]:
        print(f"  {member}: geohash={geohash[0]}  coords=({pos[0][1]:.4f}, {pos[0][0]:.4f})")

print()
print("💡 Notice drivers close together share a longer common prefix in their geohash.")
print("   This is how Redis knows which drivers to check without scanning all of them.")

## 🏗️ Building the Match Function

Let's combine everything into a function that simulates what happens when a rider requests a ride. The matching algorithm:

1. Use Redis GEOSEARCH to find nearby drivers (fast)
2. Filter to only available drivers
3. Return the ranked list (closest first)

In [ ]:
def find_nearby_drivers(rider_lng, rider_lat, radius_km=5, max_results=5):
    """
    Find the nearest available drivers to a rider's location.
    Uses Redis Geo for fast proximity search.
    
    Returns a list of (driver_key, distance_km) tuples.
    """
    r = get_redis()
    
    # Step 1: GEOSEARCH for drivers within radius
    nearby = r.geosearch(
        name="drivers:locations",
        longitude=rider_lng,
        latitude=rider_lat,
        radius=radius_km,
        unit="km",
        withdist=True,
        sort="ASC",
        count=max_results * 3  # fetch extra to account for filtering
    )
    
    # Step 2: filter to only available drivers
    available = r.smembers("drivers:available")
    matches = [
        (member, float(dist))
        for member, dist in nearby
        if member in available
    ]
    
    # Step 3: return top matches (already sorted by distance)
    return matches[:max_results]


# Test it!
print("🚗 Scenario 1: Rider in Downtown SF")
matches = find_nearby_drivers(-122.4194, 37.7749)
for driver, dist in matches:
    print(f"   {driver}: {dist:.2f} km away")

print()
print("🚗 Scenario 2: Rider near Outer Sunset (fewer drivers)")
matches = find_nearby_drivers(-122.4900, 37.7550, radius_km=3)
for driver, dist in matches:
    print(f"   {driver}: {dist:.2f} km away")
if not matches:
    print("   ❌ No drivers found! Expanding search radius...")
    matches = find_nearby_drivers(-122.4900, 37.7550, radius_km=8)
    for driver, dist in matches:
        print(f"   {driver}: {dist:.2f} km away")

## 🧹 Cleanup

In [ ]:
r = get_redis()
r.delete("drivers:locations", "drivers:available")
print("🧹 Cleaned up Redis keys")

## 📚 Summary

### Key Takeaways

1. **Regular indexes can't do proximity search** — B-trees work for one dimension, not two
2. **PostGIS** adds spatial indexes (GiST) to PostgreSQL — great for durable storage and analytics
3. **Redis Geo** uses geohashing for in-memory proximity searches — 10–50× faster than PostGIS
4. **Uber uses both** — Redis for real-time matching, PostGIS/similar for historical data
5. **GEOSEARCH** is the key Redis command — finds nearby members by radius, sorted by distance

### How This Fits in a System Design Interview

When asked "how do you find nearby drivers?", the progression is:
- ❌ Naive: scan all drivers and calculate distances → O(n) per request
- ✅ Good: PostGIS with spatial index → efficient disk-based proximity search  
- ✅ Great: Redis Geo → in-memory, handles 2M updates/sec + fast proximity search

### Next Up

In **Notebook 2**, we'll tackle **real-time driver tracking** — how to handle millions of location updates per second and keep the geo index fresh.